# 02 · Análisis exploratorio (EDA)

**Proyecto:** The AI Power Map · **Equipo:** DataChasquiAILab

**Objetivo:** explorar la distribución de los modelos por año, país, dominio, tipo de organización y accesibilidad, y la relación entre variables numéricas, para decidir qué gráficos usar en el dashboard.

| Entradas | Salidas |
|---|---|
| `data/processed/all_ai_models_clean.csv` | Tablas y gráficos exploratorios (sin archivos exportados) |

## Carga del dataset limpio

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Cargamos el dataset limpio (capa Silver)
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import CLEAN_DATASET
from src.io_utils import load_csv

df = load_csv(CLEAN_DATASET)

print(f" Dataset cargado")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Periodo: {int(df['Year'].min())} – {int(df['Year'].max())}")

 Dataset cargado
Filas: 3405
Columnas: 22
Periodo: 1950 – 2026


## 3.1 Distribución por años

In [3]:
print("="*60)
print("3.1 DISTRIBUCIÓN TEMPORAL DE MODELOS DE IA")
print("="*60)

# Filtramos desde 2000 para el análisis principal
# Los datos anteriores son muy escasos y distorsionan la escala
df_moderno = df[df['Year'] >= 2000].copy()

modelos_por_anio = df_moderno.groupby('Year').size().reset_index(name='Modelos')

print(modelos_por_anio.to_string(index=False))
print(f"\nAño con más modelos: {modelos_por_anio.loc[modelos_por_anio['Modelos'].idxmax(), 'Year']:.0f}")
print(f"Máximo de modelos en un año: {modelos_por_anio['Modelos'].max()}")
print(f"\nModelos antes del 2000: {len(df[df['Year'] < 2000])}")
print(f"Modelos desde 2000: {len(df_moderno)}")

3.1 DISTRIBUCIÓN TEMPORAL DE MODELOS DE IA
   Year  Modelos
2000.00        5
2001.00        4
2002.00        6
2003.00       10
2004.00        7
2005.00        8
2006.00       17
2007.00       10
2008.00       12
2009.00       20
2010.00       19
2011.00       17
2012.00       23
2013.00       29
2014.00       44
2015.00       43
2016.00       92
2017.00       91
2018.00       98
2019.00      168
2020.00      126
2021.00      208
2022.00      230
2023.00      512
2024.00      917
2025.00      561
2026.00       30

Año con más modelos: 2024
Máximo de modelos en un año: 917

Modelos antes del 2000: 98
Modelos desde 2000: 3307


## 3.2 Distribución por país

In [4]:
print("="*60)
print("3.2 DISTRIBUCIÓN DE MODELOS POR PAÍS")
print("="*60)

modelos_por_pais = (df.groupby('Country (of organization)')
                   .size()
                   .reset_index(name='Modelos')
                   .sort_values('Modelos', ascending=False))

# Top 15 países
top15 = modelos_por_pais.head(15)
print("Top 15 países por número de modelos:")
print(top15.to_string(index=False))

# Participación de EE.UU. y China
total = modelos_por_pais['Modelos'].sum()
usa = modelos_por_pais[modelos_por_pais['Country (of organization)'] == 'United States of America']['Modelos'].values[0]
china = modelos_por_pais[modelos_por_pais['Country (of organization)'] == 'China']['Modelos'].values[0]

print(f"\nTotal de modelos: {total}")
print(f"EE.UU.: {usa} modelos — {usa/total*100:.1f}% del total")
print(f"China: {china} modelos — {china/total*100:.1f}% del total")
print(f"EE.UU. + China: {(usa+china)/total*100:.1f}% del total")
print(f"\nResto del mundo: {total-usa-china} modelos — {(total-usa-china)/total*100:.1f}% del total")

3.2 DISTRIBUCIÓN DE MODELOS POR PAÍS
Top 15 países por número de modelos:
                           Country (of organization)  Modelos
                            United States of America     1698
                                               China      809
United Kingdom of Great Britain and Northern Ireland      179
                                              Canada      105
                                 Korea (Republic of)       77
                                             Germany       74
                                              France       72
                                               Japan       53
                                         Switzerland       49
                                           Hong Kong       38
                                              Russia       33
                                              Israel       30
                                           Singapore       28
                                            Thailand      

## 3.3 Distribución por dominio

In [5]:
print("="*60)
print("3.3 DISTRIBUCIÓN DE MODELOS POR DOMINIO")
print("="*60)

modelos_por_dominio = (df.groupby('Domain')
                      .size()
                      .reset_index(name='Modelos')
                      .sort_values('Modelos', ascending=False))

total = modelos_por_dominio['Modelos'].sum()

modelos_por_dominio['Porcentaje (%)'] = (modelos_por_dominio['Modelos'] / total * 100).round(1)

print(modelos_por_dominio.to_string(index=False))

print(f"\nTotal de modelos: {total}")
print(f"Dominios únicos: {modelos_por_dominio['Domain'].nunique()}")

3.3 DISTRIBUCIÓN DE MODELOS POR DOMINIO
           Domain  Modelos  Porcentaje (%)
         Language     1648           48.40
           Vision      385           11.30
          Biology      384           11.30
       Multimodal      195            5.70
 Image generation      183            5.40
           Speech      128            3.80
            Video      120            3.50
         Robotics       64            1.90
            Games       61            1.80
          Unknown       40            1.20
   Recommendation       34            1.00
            Other       32            0.90
            Audio       27            0.80
      Mathematics       23            0.70
      3D modeling       19            0.60
Materials science       17            0.50
    Earth science       17            0.50
         Medicine       16            0.50
          Driving        7            0.20
           Search        3            0.10
        Astronomy        1            0.00
       Psychol

## 3.4 Distribución por tipo de organización

In [6]:
print("="*60)
print("3.4 DISTRIBUCIÓN POR TIPO DE ORGANIZACIÓN")
print("="*60)

# Distribución general
org_dist = (df.groupby('Organization categorization')
            .size()
            .reset_index(name='Modelos')
            .sort_values('Modelos', ascending=False))

total = org_dist['Modelos'].sum()
org_dist['Porcentaje (%)'] = (org_dist['Modelos'] / total * 100).round(1)

print("Distribución general:")
print(org_dist.to_string(index=False))

# Evolución por década
print(f"\nEvolución por década:")
df['Decade'] = (df['Year'] // 10 * 10).astype('Int64')

decada_org = (df.groupby(['Decade', 'Organization categorization'])
              .size()
              .reset_index(name='Modelos'))

decada_pivot = decada_org.pivot(index='Decade',
                                 columns='Organization categorization',
                                 values='Modelos').fillna(0).astype(int)

print(decada_pivot.to_string())

3.4 DISTRIBUCIÓN POR TIPO DE ORGANIZACIÓN
Distribución general:
Organization categorization  Modelos  Porcentaje (%)
                   Industry     2118           62.20
                   Academia     1174           34.50
        Research collective       66            1.90
                 Government       31            0.90
                    Unknown       16            0.50

Evolución por década:
Organization categorization  Academia  Government  Industry  Research collective  Unknown
Decade                                                                                   
1950                                6           0         3                    0        0
1960                               11           0         3                    0        0
1970                                5           0         2                    0        0
1980                               23           0         6                    0        0
1990                               23           2      

## 3.5 Distribución por accesibilidad

In [7]:
print("="*60)
print("3.5 DISTRIBUCIÓN DE ACCESIBILIDAD DE MODELOS")
print("="*60)

# Accesibilidad general
access_dist = (df.groupby('Model accessibility')
               .size()
               .reset_index(name='Modelos')
               .sort_values('Modelos', ascending=False))

total = access_dist['Modelos'].sum()
access_dist['Porcentaje (%)'] = (access_dist['Modelos'] / total * 100).round(1)

print("Distribución general de accesibilidad:")
print(access_dist.to_string(index=False))

# Evolución por década
print(f"\nEvolución de accesibilidad por década:")
access_decada = (df.groupby(['Decade', 'Model accessibility'])
                 .size()
                 .reset_index(name='Modelos'))

access_pivot = access_decada.pivot(index='Decade',
                                    columns='Model accessibility',
                                    values='Modelos').fillna(0).astype(int)

print(access_pivot.to_string())

# Open vs Closed
print(f"\nResumen Open vs Closed:")
df['Access_simple'] = df['Model accessibility'].apply(
    lambda x: 'Abierto' if 'Open' in str(x) else ('Cerrado' if x == 'Unreleased' or x == 'API access' else 'Unknown')
)

access_simple = (df.groupby('Access_simple')
                 .size()
                 .reset_index(name='Modelos'))
access_simple['Porcentaje (%)'] = (access_simple['Modelos'] / total * 100).round(1)
print(access_simple.to_string(index=False))

3.5 DISTRIBUCIÓN DE ACCESIBILIDAD DE MODELOS
Distribución general de accesibilidad:
          Model accessibility  Modelos  Porcentaje (%)
                   Unreleased      823           24.20
                      Unknown      797           23.40
  Open weights (unrestricted)      788           23.10
                   API access      365           10.70
Open weights (restricted use)      291            8.50
Open weights (non-commercial)      224            6.60
       Hosted access (no API)      117            3.40

Evolución de accesibilidad por década:
Model accessibility  API access  Hosted access (no API)  Open weights (non-commercial)  Open weights (restricted use)  Open weights (unrestricted)  Unknown  Unreleased
Decade                                                                                                                                                                 
1950                          0                       0                              0              

## 3.6 Relación entre variables numéricas

In [8]:
print("="*60)
print("3.6 RELACIÓN ENTRE VARIABLES NUMÉRICAS")
print("="*60)

# Variables numéricas disponibles
numericas = ['Parameters', 'Training compute (FLOP)',
             'Training dataset size (total)',
             'Training time (hours)',
             'Training compute cost (2023 USD)',
             'Hardware quantity', 'Citations']

# Estadísticas descriptivas
print("Estadísticas descriptivas:")
print(df[numericas].describe().round(2).to_string())

# Cobertura real de cada variable
print(f"\nCobertura (% de filas con dato):")
for col in numericas:
    cobertura = (df[col].notna().sum() / len(df) * 100).round(1)
    print(f"  {col}: {cobertura}%")

# Correlación entre variables numéricas
print(f"\nMatriz de correlación (solo filas con datos completos):")
corr = df[numericas].corr(method='spearman').round(2)
print(corr.to_string())

3.6 RELACIÓN ENTRE VARIABLES NUMÉRICAS
Estadísticas descriptivas:
              Parameters        Training compute (FLOP)  Training dataset size (total)  Training time (hours)  Training compute cost (2023 USD)  Hardware quantity  Citations
count            2243.00                        1366.00                        1379.00                 544.00                            224.00             838.00    1462.00
mean     178331416051.32   1668302131449637642436608.00               2498674211312.43                 462.76                        6015289.65             980.02    4295.64
std     4250038458334.00  19770234836634406769131520.00              23517052209467.89                 881.35                       37626531.09            7985.43   15035.54
min                10.00                          40.00                           0.00                   0.10                              3.90               1.00       0.00
25%         100000000.00        13441664496072493056.00         